# Complete Row-Level Gait Classification

This notebook uses the complete combined dataset as individual sensor rows. No windowing, sequence slicing, or window-derived feature extraction is applied. It first evaluates the listed machine-learning models, then trains and evaluates a deep-learning model, and finally prints a comparison table.

In [ ]:
from pathlib import Path
import warnings
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score

from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
if torch.cuda.is_available():
    x = torch.randn(2000, 2000, device="cuda")
    y = torch.randn(2000, 2000, device="cuda")
    z = x @ y

    print("GPU test successful")
    print("Device:", z.device)

    del x, y, z
    torch.cuda.empty_cache()
else:
    print("CUDA is not available. Using CPU.")

In [ ]:
FEATURE_COLUMNS = ["X", "Y", "Z"]
TARGET_COLUMN = "Subject"

candidate_paths = [
    Path.cwd() / "right_pocket_sessions_1_2_combined.csv",
    Path.cwd() / "Combined_RightPocket" / "right_pocket_sessions_1_2_combined.csv",
]

DATA_PATH = next(
    (path for path in candidate_paths if path.is_file()),
    None
)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find right_pocket_sessions_1_2_combined.csv"
    )

raw_df = pd.read_csv(
    DATA_PATH,
    usecols=FEATURE_COLUMNS + [TARGET_COLUMN],
    dtype={
        "X": "float32",
        "Y": "float32",
        "Z": "float32",
        TARGET_COLUMN: "int32"
    }
)

before = len(raw_df)

raw_df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

raw_df.dropna(
    subset=FEATURE_COLUMNS + [TARGET_COLUMN],
    inplace=True
)

if raw_df.empty:
    raise ValueError("No valid rows remain.")

if len(raw_df) != before:
    print(f"Removed invalid rows: {before - len(raw_df):,}")

X = raw_df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(
    raw_df[TARGET_COLUMN].to_numpy()
).astype(np.int64)

if len(label_encoder.classes_) < 2:
    raise ValueError("At least two subject classes are required.")

print(f"Dataset: {DATA_PATH}")
print(f"Rows: {len(X):,}")
print(f"Subjects: {len(label_encoder.classes_)}")
print(f"Features: {X.shape[1]}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y
)

scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train_raw).astype(
    np.float32,
    copy=False
)

X_test = scaler.transform(X_test_raw).astype(
    np.float32,
    copy=False
)

del X_train_raw, X_test_raw

print(f"Training rows: {len(X_train):,}")
print(f"Testing rows: {len(X_test):,}")
print(f"Training shape: {X_train.shape}")
print(f"Testing shape: {X_test.shape}")

## Machine-Learning Models

In [ ]:
models = {
    "Naive Bayes": GaussianNB(),

    "Linear SVM": LinearSVC(
        C=1.0,
        max_iter=1000,
        dual="auto",
        random_state=RANDOM_SEED
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_SEED,
        max_depth=20
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        random_state=RANDOM_SEED,
        n_jobs=1
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=100,
        max_depth=20,
        random_state=RANDOM_SEED,
        n_jobs=1
    )
}

if XGBClassifier is not None:
    models["XGBoost"] = XGBClassifier(
        n_estimators=100,
        max_depth=8,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=1.0,
        tree_method="hist",
        eval_metric="mlogloss",
        random_state=RANDOM_SEED,
        n_jobs=-1
    )

print("Models:")
for name in models:
    print("-", name)

In [ ]:
train_test_results = {}
train_times = {}

MAX_TREE_TRAIN = 500_000

rng = np.random.default_rng(RANDOM_SEED)

if len(X_train) > MAX_TREE_TRAIN:
    idx = rng.choice(
        len(X_train),
        size=MAX_TREE_TRAIN,
        replace=False
    )
    X_tree = X_train[idx]
    y_tree = y_train[idx]
else:
    X_tree = X_train
    y_tree = y_train

print(f"Tree models using {len(X_tree):,} training rows")

for name, model in models.items():

    if name in ["Random Forest", "Extra Trees", "XGBoost", "Decision Tree"]:
        X_fit = X_tree
        y_fit = y_tree
    else:
        X_fit = X_train
        y_fit = y_train

    print(f"\nTraining {name}...")

    start = time.time()

    model.fit(X_fit, y_fit)

    train_time = time.time() - start

    print(f"Training time: {train_time:.2f} seconds")

    start = time.time()

    PREDICT_BATCH_SIZE = 5_000

    y_pred = np.concatenate([
        model.predict(X_test[batch_start:batch_start + PREDICT_BATCH_SIZE])
        for batch_start in range(0, X_test.shape[0], PREDICT_BATCH_SIZE)
    ])

    predict_time = time.time() - start

    acc = accuracy_score(y_test, y_pred)

    train_test_results[name] = acc
    train_times[name] = train_time

    print(f"Prediction time: {predict_time:.2f} seconds")
    print(f"Accuracy: {acc:.4f}")

del X_tree, y_tree

In [ ]:
# 6. K-FOLD VALIDATION FOR MACHINE-LEARNING MODELS

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
kfold_results = {}

print("\n===== K-FOLD RESULTS =====")

for name, model in models.items():
    accuracies = []

    for train_idx, test_idx in skf.split(X_scaled, y):
        X_train_k, X_test_k = X_scaled[train_idx], X_scaled[test_idx]
        y_train_k, y_test_k = y.iloc[train_idx], y.iloc[test_idx]

        fold_ros = RandomOverSampler(random_state=RANDOM_SEED)
        X_train_k_res, y_train_k_res = fold_ros.fit_resample(X_train_k, y_train_k)

        fold_model = clone(model)
        fold_model.fit(X_train_k_res, y_train_k_res)
        PREDICT_BATCH_SIZE = 5_000
        y_pred_k = np.concatenate([
            fold_model.predict(X_test_k[batch_start:batch_start + PREDICT_BATCH_SIZE])
            for batch_start in range(0, X_test_k.shape[0], PREDICT_BATCH_SIZE)
        ])

        accuracies.append(accuracy_score(y_test_k, y_pred_k))

    avg_acc = np.mean(accuracies)
    kfold_results[name] = avg_acc

    print(f"{name}: {avg_acc:.4f}")


## Deep-Learning Model

In [ ]:
# 7. TRAIN DEEP-LEARNING MODEL AFTER MACHINE-LEARNING MODELS

BATCH_SIZE = 4096
EPOCHS = 50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_dataset = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_train.to_numpy(dtype=np.int64)),
)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=device.type == "cuda",
)

deep_model = nn.Sequential(
    nn.Linear(len(FEATURE_COLUMNS), 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, len(label_encoder.classes_)),
).to(device)

loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(deep_model.parameters(), lr=1e-3)

print(f"Training device: {device}")

for epoch in range(EPOCHS):
    deep_model.train()
    total_loss = 0.0
    trained_rows = 0

    for batch_features, batch_labels in train_loader:
        batch_features = batch_features.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = deep_model(batch_features)
        loss = loss_function(logits, batch_labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(batch_labels)
        trained_rows += len(batch_labels)

    print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {total_loss / trained_rows:.4f}")


In [ ]:
# 8. EVALUATE DEEP-LEARNING MODEL

test_dataset = TensorDataset(
    torch.from_numpy(X_test),
    torch.from_numpy(y_test.to_numpy(dtype=np.int64)),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=device.type == "cuda",
)

deep_model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        batch_features = batch_features.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)
        predictions = deep_model(batch_features).argmax(dim=1)
        correct += (predictions == batch_labels).sum().item()
        total += len(batch_labels)

deep_learning_accuracy = correct / total

print(f"Deep Learning: {deep_learning_accuracy:.4f}")
print(f"Row-level test accuracy: {deep_learning_accuracy:.4f} ({deep_learning_accuracy:.2%})")
print(f"Correct test rows: {correct:,} / {total:,}")


## Final Comparison

In [ ]:
# 9. FINAL COMPARISON TABLE

comparison_df = pd.DataFrame({
    "Model": list(models.keys()) + ["Deep Learning"],
    "Train-Test Accuracy": [train_test_results[m] for m in models.keys()] + [deep_learning_accuracy],
    "K-Fold Accuracy": [kfold_results[m] for m in models.keys()] + [np.nan],
})

print("\n===== FINAL COMPARISON =====")
print(comparison_df)
comparison_df


In [ ]:
# 10. PLOT MODEL ACCURACY COMPARISON

comparison_df.set_index("Model").plot(kind="bar", figsize=(12, 6))
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


**Split note:** this is a random, stratified split of individual rows as requested. Adjacent sensor rows may therefore appear in both partitions; the reported accuracy measures this row-level split, not generalization to unseen subjects or recording sessions.